# Import all libraries needed

In [1]:
# source .venv/bin/activate
import numpy as np
import pandas as pd
from paperscraper.pubmed import get_pubmed_papers 
from paperscraper.pubmed import get_query_from_keywords_and_date
import polars as pl

# For the loops for each time block
from datetime import datetime, timedelta
import time
import calendar
from Bio import Entrez

# Loading to db
# importing os module for environment variables
import os
# importing necessary functions from dotenv library
from dotenv import load_dotenv 
# loading variables from .env file
load_dotenv() 
import psycopg2
from sqlalchemy import create_engine

INFO:paperscraper.load_dumps:Loaded biorxiv dump with 570 entries


# Testing get_pubmed_papers

In [ ]:
eighteenseventy_twentyten_pcos = '("1870/01/01"[Date - Create] : "1880/12/30"[Date - Create] AND "cancers[Title]")'
pcos = get_pubmed_papers(query=eighteenseventy_twentyten_pcos, max_results=9998)
print(pcos.info())
print(pcos)

<class 'pandas.DataFrame'>
RangeIndex: 0 entries
Empty DataFrame
None
Empty DataFrame
Columns: []
Index: []


In [ ]:
twentyeleven_twentysix_pcos = '("2011/01/01"[Date - Create] : "2026/6/25"[Date - Create] AND "pcos[Title]")'
pcos22 = get_pubmed_papers(query=twentyeleven_twentysix_pcos, max_results=9998)
print(pcos22.info())
print(pcos22)

<class 'pandas.DataFrame'>
RangeIndex: 3113 entries, 0 to 3112
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   title     3113 non-null   str   
 1   abstract  2877 non-null   str   
 2   journal   3113 non-null   str   
 3   date      3113 non-null   str   
 4   authors   3113 non-null   object
 5   doi       2995 non-null   str   
dtypes: object(1), str(5)
memory usage: 5.4+ MB
None
                                                  title  \
0     Diagnostic and prognostic biomarkers in PCOS: ...   
1     Correction: THBS1: a biomarker for PCOS and it...   
2     Beyond association: Mediation roles of lipidom...   
3     Retraction notice to "A new look at low-dose a...   
4     A 3.6:1 myo-inositol to D-chiro-inositol ratio...   
...                                                 ...   
3108  Reply of the Authors: Luteal-phase clomiphene ...   
3109  Study of Omentin1 and Other Adipokines and Hor...   
3110  A single nu

## Brainstorm
First we will look at general health - reddit forums
Then mental health - reddit forums
Then women's health - reddit forums

1. Women's health 
2. Mental Health
3. General Health

1950 - June 19th 2026

Example usage
endo_papers = get_pubmed_papers(query="endometriosis", max_results=10)

In [ ]:
date_query = '("2026/06/24"[Date - Create])'
today = get_pubmed_papers(query=date_query, max_results=9998)
print(today.info())

<class 'pandas.DataFrame'>
RangeIndex: 5740 entries, 0 to 5739
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   title     5740 non-null   str   
 1   abstract  5030 non-null   str   
 2   journal   5740 non-null   str   
 3   date      5740 non-null   str   
 4   authors   5740 non-null   object
 5   doi       5721 non-null   str   
dtypes: object(1), str(5)
memory usage: 9.5+ MB
None


In [113]:
date_query = '("1870/01/1"[Date - Create] : "1900/12/31"[Date - Create] AND "cancers[Title/abstract]")'
today = get_pubmed_papers(query=date_query, max_results=9998)
print(today.info())
print(today)

<class 'pandas.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   title     2 non-null      str   
 1   abstract  0 non-null      object
 2   journal   2 non-null      str   
 3   date      2 non-null      str   
 4   authors   2 non-null      object
 5   doi       1 non-null      str   
dtypes: object(2), str(4)
memory usage: 427.0+ bytes
None
                                               title abstract  \
0                             Cancers of the Larynx.     None   
1  The Infection of the Connective Tissue in Scir...     None   

                             journal        date       authors  \
0                  Annals of surgery  1889-05-01     [J BBall]   
1  Journal of anatomy and physiology  1879-10-01  [CCreighton]   

                                doi  
0  10.1097/00000658-188901000-00117  
1                               NaN  


# Create list of final keyword query taken from a womens health study hospital

In [13]:
# womens health condition query
# source: https://www.brighamandwomens.org/womens-health/diseases-conditions
conditions = ["Abdominal cerclage", "Abnormal pap smears", "Abnormal uterine bleeding",  "Alzheimer's disease", "Amenorrhea",
            "Amniocentesis", "Arthritis", "Asthma", "Bacterial vaginosis", "Bleeding disorders", "Breast cancer", "Cancers",
            "Celiac disease", "Cervical cancer", "Cervical insufficiency", "Cervicitis", "Cesarean section (C-section)",
            "Chorionic villus sampling", "Colposcopy", "Congenital abnormalities", "Continence", "Contraception (birth control)",
            "Cystoscopy", "Depression and anxiety", "Dilation and curettage (D and C)", "Dysmenorrhea",  "Endometrial ablation",  "Endometrial biopsy",
            "Endometrial cancer", "Endometriosis", "Epilepsy", "Family planning", "Female genital cutting", "Fertility preservation",
            "Fibroma", "Gallstone disease",  "Genetic conditions", "Gentle cesarean section", "Gestational diabetes", "Gestational trophoblastic disease",
            "Graves' disease", "Gynecologic cancer", "Gynecologic specialty care", "Gynecologic well-woman care", "Gynecological surgery",  "Heart disease",
             "Heavy menstrual cycles", "High-risk pregnancy","HIV", "HPV", "In-vitro fertilization", "Infertility", "Integrative medicine", "Interstitial cystitis",
             "Irritable bowel syndrome (IBS)", "Laparoscopic hysterectomy", "Laparoscopy", "Loop electrosurgical excision procedure (LEEP)", "Lupus",
             "Menopause", "Menorrhagia", "Menstrual conditions", "Miscarriage", "Morning sickness", "Multiple sclerosis", "Myomectomy", "Osteoarthritis", "Osteoporosis",
             "Ovarian cancer", "Ovarian cysts", "Ovarian fibroma", "Overactive bladder", "Pancreatic cystic neoplasm", "Pap test", "Pelvic inflammatory disease (PID)",
             "Pelvic organ prolapse", "Pelvic pain", "Pelvic ultrasound", "Perimenopause", "Placenta accreta", "Placenta previa", "Polycystic ovary syndrome (PCOS)",
             "Postpartum depression", "Preconception planning", "Pregnancy", "Pregnancy complications", "Pregnancy, first trimester", "Pregnancy, multiples",
             "Pregnancy, second trimester", "Pregnancy, third trimester", "Premenstrual dysphoric disorder (PMDD)", "Premenstrual syndrome (PMS)", "Prenatal care", "Prenatal genetic screening", "Preterm birth",
             "Primary care for women", "Recurrent pregnancy loss", "Rheumatoid arthritis", "Robotic hysterectomy", "Robotic myomectomy", "Sexually transmitted diseases (STDs)", 
             "Sleep disorders", "Small vessel disease", "Sports injuries", "STIs", "Stress urinary incontinence", "Stroke", "Thyroid disease", "Transgender care", "Tubal ligation",
             "Turner syndrome", "Type I autoimmune hepatitis", "Ultrasound", "Urinary incontinence", "Urinary tract infections (UTIs)", "Uterine and vaginal prolapse", "Uterine fibroids",
             "Vaginal birth after cesarean section", "Vaginal cancer", "Vaginitis", "Vulvar cancer", "Vulvar dysplasia", "Vulvitis", "Yeast infection"
]

example = ["Abdominal cerclage", "Abnormal pap smears", "Abnormal uterine bleeding"]

# These conditions fall under these categories
'''
  Reproductive & Gynecologic, Cancers & Oncology, Pregnancy & Obstetrics, Fertility & Family Planning, Procedures & Tests,
  Hormonal & Endocrine, Urogynecologic, Neurological, Musculoskeletal, Cardiovascular & Vascular, Gastrointestinal,
  Infectious Disease & Sexual Health, Mental Health, Autoimmune, and Other / Care & Services
'''

# Critical Areas for Womens Health Research
# https://axiawh.com/resources/womens-health-is-under-researched/
'''
Breast Cancer, Endometriosis, Sexual Health, Postpartum Hypertension,
'''


'\nBreast Cancer, Endometriosis, Sexual Health, Postpartum Hypertension,\n'

In [ ]:
date_query = f'("2026/06/25"[Date - Create] AND ({example}))'
today = get_pubmed_papers(query=date_query, max_results=9998)
print(today)

Empty DataFrame
Columns: []
Index: []


In [ ]:
blood = '("2026/06/01"[Date - Create] : "2026/6/24"[Date - Create] AND (blood[Title/abstract]))'
blood_results = get_pubmed_papers(query=blood, max_results=9998)

In [30]:
print(blood_results.info())

<class 'pandas.DataFrame'>
RangeIndex: 7268 entries, 0 to 7267
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   title     7268 non-null   str   
 1   abstract  7148 non-null   str   
 2   journal   7260 non-null   str   
 3   date      7268 non-null   str   
 4   authors   7268 non-null   object
 5   doi       7222 non-null   str   
dtypes: object(1), str(5)
memory usage: 14.9+ MB
None


In [58]:
keywords = ["Abdominal cerclage", "Abnormal pap smears", "Abnormal uterine bleeding",  "Alzheimer's disease", "Amenorrhea",
            "Amniocentesis", "Arthritis", "Asthma", "Bacterial vaginosis", "Bleeding disorders", "Breast cancer",
            "Celiac disease", "Cervical cancer", "Cervical insufficiency", "Cervicitis", "Cesarean section (C-section)"]

date_women = '("2026/06/01"[Date - Create] : "2026/6/24"[Date - Create])'
keywords_women = " OR ".join([f"{word}[Title]" for word in keywords])

date_key = f"({date_women}) AND {keywords_women})"

women_test = get_pubmed_papers(query=date_key, max_results=9998)
print(women_test.info())

INFO:pymed_paperscraper.api:ChunkedEncodingError: Response ended prematurely.	Now at 1 attempts for ['40537290', '40537268', '40535870', '40535416', '40535129', '40533823', '40533618', '40532888', '40532887', '40531956', '40531303', '40530360', '40530076', '40528439', '40528133', '40528071', '40527062', '40525055', '40525053', '40524947', '40524677', '40524241', '40524156', '40523495', '40522936', '40522505', '40522274', '40521792', '40521575', '40521260', '40520651', '40520265', '40519570', '40519226', '40519038', '40518987', '40518467', '40517682', '40517207', '40516128', '29999857', '40515671', '40515538', '40513496', '40513284', '40512829', '40512717', '40512138', '40511424', '40511255', '40510482', '40510096', '40509826', '40507506', '40506958', '40506370', '40506343', '40505985', '40505450', '40505447', '40505432', '40505112', '40505101', '40505037', '40503342', '40502965', '40502651', '40500831', '40499527', '40498141', '40498055', '40497795', '40497566', '40496321', '40495474',

In [ ]:
keywords = ["Abdominal cerclage", "Abnormal pap smears", "Abnormal uterine bleeding",  "Alzheimer's disease", "Amenorrhea",
            "Amniocentesis", "Arthritis", "Asthma", "Bacterial vaginosis", "Bleeding disorders", "Breast cancer", "Cancers",
            "Celiac disease", "Cervical cancer", "Cervical insufficiency", "Cervicitis", "Cesarean section (C-section)",
            "Chorionic villus sampling", "Colposcopy", "Congenital abnormalities", "Continence", "Contraception (birth control)",
            "Cystoscopy", "Depression and anxiety", "Dilation and curettage (D and C)", "Dysmenorrhea",  "Endometrial ablation",  "Endometrial biopsy",
            "Endometrial cancer", "Endometriosis", "Epilepsy", "Family planning", "Female genital cutting", "Fertility preservation",
            "Fibroma", "Gallstone disease",  "Genetic conditions", "Gentle cesarean section", "Gestational diabetes", "Gestational trophoblastic disease",
            "Graves' disease", "Gynecologic cancer", "Gynecologic specialty care", "Gynecologic well-woman care", "Gynecological surgery",  "Heart disease",
             "Heavy menstrual cycles", "High-risk pregnancy","HIV", "HPV", "In-vitro fertilization", "Infertility", "Integrative medicine", "Interstitial cystitis",
             "Irritable bowel syndrome (IBS)", "Laparoscopic hysterectomy", "Laparoscopy", "Loop electrosurgical excision procedure (LEEP)", "Lupus",
             "Menopause", "Menorrhagia", "Menstrual conditions", "Miscarriage", "Morning sickness", "Multiple sclerosis", "Myomectomy", "Osteoarthritis", "Osteoporosis",
             "Ovarian cancer", "Ovarian cysts", "Ovarian fibroma", "Overactive bladder", "Pancreatic cystic neoplasm", "Pap test", "Pelvic inflammatory disease (PID)",
             "Pelvic organ prolapse", "Pelvic pain", "Pelvic ultrasound", "Perimenopause", "Placenta accreta", "Placenta previa", "Polycystic ovary syndrome (PCOS)",
             "Postpartum depression", "Preconception planning", "Pregnancy", "Pregnancy complications", "Pregnancy, first trimester", "Pregnancy, multiples",
             "Pregnancy, second trimester", "Pregnancy, third trimester", "Premenstrual dysphoric disorder (PMDD)", "Premenstrual syndrome (PMS)", "Prenatal care", "Prenatal genetic screening", "Preterm birth",
             "Primary care for women", "Recurrent pregnancy loss", "Rheumatoid arthritis", "Robotic hysterectomy", "Robotic myomectomy", "Sexually transmitted diseases (STDs)", 
             "Sleep disorders", "Small vessel disease", "Sports injuries", "STIs", "Stress urinary incontinence", "Stroke", "Thyroid disease", "Transgender care", "Tubal ligation",
             "Turner syndrome", "Type I autoimmune hepatitis", "Ultrasound", "Urinary incontinence", "Urinary tract infections (UTIs)", "Uterine and vaginal prolapse", "Uterine fibroids",
             "Vaginal birth after cesarean section", "Vaginal cancer", "Vaginitis", "Vulvar cancer", "Vulvar dysplasia", "Vulvitis", "Yeast infection"
]

# Looping scripts to retrieve data between each time block for every keyword
## Keyword1 List and Keyword2 List

### First time block: 1870 - 1983

In [148]:
# FIRST SET OF KEYWORDS 
# PER YEAR
######## 1870 - 1983 ###########

keywords_df1 = ["Abdominal cerclage", "Abnormal pap smears", "Abnormal uterine bleeding",  "Alzheimer's disease", "Amenorrhea",
            "Amniocentesis", "Arthritis", "Asthma", "Bacterial vaginosis", "Bleeding disorders", "Breast cancer", "Cancers",
            "Celiac disease", "Cervical cancer", "Cervical insufficiency", "Cervicitis", "Cesarean section (C-section)",
            "Chorionic villus sampling", "Colposcopy", "Congenital abnormalities", "Continence", "Contraception (birth control)",
            "Cystoscopy", "Depression and anxiety", "Dilation and curettage (D and C)", "Dysmenorrhea",  "Endometrial ablation",  "Endometrial biopsy",
            "Endometrial cancer", "Endometriosis", "Epilepsy", "Family planning", "Female genital cutting", "Fertility preservation",
            "Fibroma", "Gallstone disease",  "Genetic conditions", "Gentle cesarean section", "Gestational diabetes", "Gestational trophoblastic disease",
            "Graves' disease", "Gynecologic cancer", "Gynecologic specialty care", "Gynecologic well-woman care", "Gynecological surgery",  "Heart disease",
             "Heavy menstrual cycles", "High-risk pregnancy","HIV", "HPV", "In-vitro fertilization", "Infertility", "Integrative medicine", "Interstitial cystitis",
             "Irritable bowel syndrome (IBS)", "Laparoscopic hysterectomy", "Laparoscopy", "Loop electrosurgical excision procedure (LEEP)"
]

Entrez.email = "sebrodnick@willamette.edu"  # this identifies who is pulling and lessen 429 errors for pulling

keywords_joined1 = " OR ".join(f'"{word}"[Title/abstract]' for word in keywords_df1)

start_date1 = datetime(1870, 1, 1)
end_date1 = datetime(1983, 12, 31)

# List to hold dfs for each day
dfs_list1 = []
# Loop through each day
current_date1 = start_date1

while current_date1 <= end_date1:

    #date_str1 = current_date1.strftime("%Y/%m/%d") # pubmed needs YYYY/MM/DD
    #day_query1 = f'("{date_str1}"[Date - Create]) AND ({keywords_joined1})'

    start_str1 = f"{current_date1.year}/01/01"
    end_str1 = f"{current_date1.year}/12/31"
    year_query1 = f'("{start_str1}"[Date - Create] : "{end_str1}"[Date - Create]) AND ({keywords_joined1})'

    try:
        df1 = get_pubmed_papers(query = year_query1, max_results = 9998)
        if df1 is not None and not df1.empty:
            dfs_list1.append(df1)
            print(f"Papers found for this year {start_str1} - {end_str1}: {len(df1)}")
        else:
            print(f"No papers found for this year {start_str1} - {end_str1}")
        
    except Exception as e:
        print(f"Error getting papers for {current_date1.year}: {e}")

    time.sleep(3) # This will pause (3 seconds) between each loop, this will also help with 429 errors 
    current_date1 = datetime(current_date1.year + 1, 1, 1)

    # Move to next day
   #current_date1 += timedelta(days=1)

   # Move to the next month
   # if current_date1.month == 12:
        #current_date1 = datetime(current_date1.year + 1, 1, 1)
    #else:
        #current_date1 = datetime(current_date1.year, current_date1.month + 1, 1)


if dfs_list1:
    final_df1 = pd.concat(dfs_list1, ignore_index=True)
    print(f"Total number of papers: {len(final_df1)}")

else:
    print("No data collected")
    final_df1 = pd.DataFrame()


No papers found for this year 1870/01/01 - 1870/12/31
No papers found for this year 1871/01/01 - 1871/12/31
No papers found for this year 1872/01/01 - 1872/12/31
No papers found for this year 1873/01/01 - 1873/12/31
No papers found for this year 1874/01/01 - 1874/12/31
No papers found for this year 1875/01/01 - 1875/12/31
No papers found for this year 1876/01/01 - 1876/12/31
No papers found for this year 1877/01/01 - 1877/12/31
No papers found for this year 1878/01/01 - 1878/12/31
Papers found for this year 1879/01/01 - 1879/12/31: 1
No papers found for this year 1880/01/01 - 1880/12/31
No papers found for this year 1881/01/01 - 1881/12/31
No papers found for this year 1882/01/01 - 1882/12/31
No papers found for this year 1883/01/01 - 1883/12/31
Papers found for this year 1884/01/01 - 1884/12/31: 1
No papers found for this year 1885/01/01 - 1885/12/31
Papers found for this year 1886/01/01 - 1886/12/31: 1
Papers found for this year 1887/01/01 - 1887/12/31: 1
Papers found for this year 1

In [142]:
print(final_df1)

                                                   title abstract  \
0      The Infection of the Connective Tissue in Scir...     None   
1      Deafness in white cats, and statistics of deaf...     None   
2                     DR. HUGHLINGS-JACKSON ON EPILEPSY.     None   
3      The Causation of Several Variations and Congen...     None   
4              Note on Electrolysis for Uterine Fibroma.     None   
...                                                  ...      ...   
28077                [On the problem of pykno-epilepsy].      NaN   
28078  [Effect of closing and opening of the eyes on ...      NaN   
28079  [Considerations on epileptic manifestations of...      NaN   
28080  [Physiopathological considerations on Hunt's d...      NaN   
28081  Histology of the liver in congenital heart dis...      NaN   

                                                 journal        date  \
0                      Journal of anatomy and physiology  1879-10-01   
1                          

In [149]:
# SECOND SET OF KEYWORDS 
# PER YEAR
######## 1870 - 1983 ###########

keywords_df2 = ["Lupus", "Menopause", "Menorrhagia", "Menstrual conditions", "Miscarriage", "Morning sickness", "Multiple sclerosis", "Myomectomy", "Osteoarthritis", "Osteoporosis",
             "Ovarian cancer", "Ovarian cysts", "Ovarian fibroma", "Overactive bladder", "Pancreatic cystic neoplasm", "Pap test", "Pelvic inflammatory disease (PID)",
             "Pelvic organ prolapse", "Pelvic pain", "Pelvic ultrasound", "Perimenopause", "Placenta accreta", "Placenta previa", "Polycystic ovary syndrome (PCOS)",
             "Postpartum depression", "Preconception planning", "Pregnancy", "Pregnancy complications", "Pregnancy, first trimester", "Pregnancy, multiples",
             "Pregnancy, second trimester", "Pregnancy, third trimester", "Premenstrual dysphoric disorder (PMDD)", "Premenstrual syndrome (PMS)", "Prenatal care", "Prenatal genetic screening", 
             "Preterm birth", "Primary care for women", "Recurrent pregnancy loss", "Rheumatoid arthritis", "Robotic hysterectomy", "Robotic myomectomy", "Sexually transmitted diseases (STDs)", 
             "Sleep disorders", "Small vessel disease", "Sports injuries", "STIs", "Stress urinary incontinence", "Stroke", "Thyroid disease", "Transgender care", "Tubal ligation",
             "Turner syndrome", "Type I autoimmune hepatitis", "Ultrasound", "Urinary incontinence", "Urinary tract infections (UTIs)", "Uterine and vaginal prolapse", "Uterine fibroids",
             "Vaginal birth after cesarean section", "Vaginal cancer", "Vaginitis", "Vulvar cancer", "Vulvar dysplasia", "Vulvitis", "Yeast infection"
]

Entrez.email = "sebrodnick@willamette.edu"  # this identifies who is pulling and lessen 429 errors for pulling

keywords_joined2 = " OR ".join(f'"{word}"[Title/abstract]' for word in keywords_df2)

start_date2 = datetime(1870, 1, 1)
end_date2 = datetime(1983, 12, 31)

# List to hold dfs for each day
dfs_list2 = []
# Loop through each day
current_date2 = start_date2

while current_date2 <= end_date2:

    #date_str1 = current_date1.strftime("%Y/%m/%d") # pubmed needs YYYY/MM/DD
    #day_query1 = f'("{date_str1}"[Date - Create]) AND ({keywords_joined1})'

    start_str2 = f"{current_date2.year}/01/01"
    end_str2 = f"{current_date2.year}/12/31"
    year_query2 = f'("{start_str2}"[Date - Create] : "{end_str2}"[Date - Create]) AND ({keywords_joined2})'

    try:
        df2 = get_pubmed_papers(query = year_query2, max_results = 9998)
        if df2 is not None and not df2.empty:
            dfs_list2.append(df2)
            print(f"Papers found for this year {start_str2} - {end_str2}: {len(df2)}")
        else:
            print(f"No papers found for this year {start_str2} - {end_str2}")
        
    except Exception as e:
        print(f"Error getting papers for {current_date2.year}: {e}")

    time.sleep(3) # This will pause (3 seconds) between each loop, this will also help with 429 errors 
    current_date2 = datetime(current_date2.year + 1, 1, 1)

    # Move to next day
   #current_date1 += timedelta(days=1)

   # Move to the next month
   # if current_date1.month == 12:
        #current_date1 = datetime(current_date1.year + 1, 1, 1)
    #else:
        #current_date1 = datetime(current_date1.year, current_date1.month + 1, 1)


if dfs_list2:
    final_df2 = pd.concat(dfs_list2, ignore_index=True)
    print(f"Total number of papers: {len(final_df2)}")

else:
    print("No data collected")
    final_df2 = pd.DataFrame()


No papers found for this year 1870/01/01 - 1870/12/31
Papers found for this year 1871/01/01 - 1871/12/31: 1
No papers found for this year 1872/01/01 - 1872/12/31
No papers found for this year 1873/01/01 - 1873/12/31
No papers found for this year 1874/01/01 - 1874/12/31
No papers found for this year 1875/01/01 - 1875/12/31
No papers found for this year 1876/01/01 - 1876/12/31
No papers found for this year 1877/01/01 - 1877/12/31
No papers found for this year 1878/01/01 - 1878/12/31
No papers found for this year 1879/01/01 - 1879/12/31
No papers found for this year 1880/01/01 - 1880/12/31
No papers found for this year 1881/01/01 - 1881/12/31
No papers found for this year 1882/01/01 - 1882/12/31
No papers found for this year 1883/01/01 - 1883/12/31
No papers found for this year 1884/01/01 - 1884/12/31
Papers found for this year 1885/01/01 - 1885/12/31: 2
No papers found for this year 1886/01/01 - 1886/12/31
Papers found for this year 1887/01/01 - 1887/12/31: 1
No papers found for this yea

In [146]:
print(final_df2)

                                                   title abstract  \
0                 Uterine Contractions during Pregnancy.     None   
1      The Washington monument, and the lightning str...     None   
2      THE WASHINGTON MONUMENT, AND THE LIGHTNING STR...     None   
3      Detachment of the Retina in both Eyes with Alb...     None   
4                                    A Lightning Stroke.     None   
...                                                  ...      ...   
30657  [Some data on the functional state of the live...      NaN   
30658  [Fetal masculinization produced by sex steroid...      NaN   
30659                           [Anemias in obstetrics].      NaN   
30660  [On pregnancy of the rudimentary accessory cor...      NaN   
30661  Some comments on the relationship of the distr...      NaN   

                                                 journal        date  \
0                      Journal of anatomy and physiology  1871-11-01   
1                          

### Second time block: 1984 - 2000

In [151]:
# FIRST SET OF KEYWORDS 
# PER MONTH
######## 1984 - 2000 ###########

keywords_df1 = ["Abdominal cerclage", "Abnormal pap smears", "Abnormal uterine bleeding",  "Alzheimer's disease", "Amenorrhea",
            "Amniocentesis", "Arthritis", "Asthma", "Bacterial vaginosis", "Bleeding disorders", "Breast cancer", "Cancers",
            "Celiac disease", "Cervical cancer", "Cervical insufficiency", "Cervicitis", "Cesarean section (C-section)",
            "Chorionic villus sampling", "Colposcopy", "Congenital abnormalities", "Continence", "Contraception (birth control)",
            "Cystoscopy", "Depression and anxiety", "Dilation and curettage (D and C)", "Dysmenorrhea",  "Endometrial ablation",  "Endometrial biopsy",
            "Endometrial cancer", "Endometriosis", "Epilepsy", "Family planning", "Female genital cutting", "Fertility preservation",
            "Fibroma", "Gallstone disease",  "Genetic conditions", "Gentle cesarean section", "Gestational diabetes", "Gestational trophoblastic disease",
            "Graves' disease", "Gynecologic cancer", "Gynecologic specialty care", "Gynecologic well-woman care", "Gynecological surgery",  "Heart disease",
             "Heavy menstrual cycles", "High-risk pregnancy","HIV", "HPV", "In-vitro fertilization", "Infertility", "Integrative medicine", "Interstitial cystitis",
             "Irritable bowel syndrome (IBS)", "Laparoscopic hysterectomy", "Laparoscopy", "Loop electrosurgical excision procedure (LEEP)"
]

Entrez.email = "sebrodnick@willamette.edu"  # this identifies who is pulling and lessen 429 errors for pulling

keywords_joined = " OR ".join(f'"{word}"[Title/abstract]' for word in keywords_df1)

start_date = datetime(1984, 1, 1)
end_date = datetime(2000, 12, 31)

# List to hold dfs for each day
dfs_list = []
# Loop through each day
current_date = start_date

while current_date <= end_date:
    last_day = calendar.monthrange(current_date.year, current_date.month)[1]
    start_str = current_date.strftime("%Y/%m/01")
    end_str = current_date.strftime(f"%Y/%m/{last_day}")
    month_query = f'("{start_str}"[Date - Create] : "{end_str}"[Date - Create]) AND ({keywords_joined})'

    try:
        df = get_pubmed_papers(query = month_query, max_results = 9998)
        if df is not None and not df.empty:
            dfs_list.append(df)
            print(f"Papers found for this year {start_str} - {end_str}: {len(df)}")

        else:
            print(f"No papers found for this month {start_str} - {end_str}")
    except Exception as e:
        print(f"Error getting papers for {start_str} - {end_str}: {e}")

    time.sleep(3) # This will pause (3 seconds) between each loop, this will also help with 429 errors 
    
    if current_date.month == 12:
        current_date = datetime(current_date.year + 1, 1, 1)
    else:
        current_date = datetime(current_date.year, current_date.month + 1, 1)

if dfs_list:
    final_df = pd.concat(dfs_list, ignore_index=True)
    print(f"Total number of papers: {len(final_df)}")

else:
    print("No data collected")
    final_df = pd.DataFrame()

Papers found for this year 1984/01/01 - 1984/01/31: 2705
Papers found for this year 1984/02/01 - 1984/02/29: 714
Papers found for this year 1984/03/01 - 1984/03/31: 679
Papers found for this year 1984/04/01 - 1984/04/30: 762
Papers found for this year 1984/05/01 - 1984/05/31: 690
Papers found for this year 1984/06/01 - 1984/06/30: 719
Papers found for this year 1984/07/01 - 1984/07/31: 706
Papers found for this year 1984/08/01 - 1984/08/31: 590
Papers found for this year 1984/09/01 - 1984/09/30: 795
Papers found for this year 1984/10/01 - 1984/10/31: 702
Papers found for this year 1984/11/01 - 1984/11/30: 678
Papers found for this year 1984/12/01 - 1984/12/31: 746
Papers found for this year 1985/01/01 - 1985/01/31: 2823
Papers found for this year 1985/02/01 - 1985/02/28: 596
Papers found for this year 1985/03/01 - 1985/03/31: 758
Papers found for this year 1985/04/01 - 1985/04/30: 714
Papers found for this year 1985/05/01 - 1985/05/31: 786
Papers found for this year 1985/06/01 - 1985/0

In [152]:
# SECOND SET OF KEYWORDS 
# PER MONTH
######## 1984 - 2000 ###########

keywords_df2 = ["Lupus", "Menopause", "Menorrhagia", "Menstrual conditions", "Miscarriage", "Morning sickness", "Multiple sclerosis", "Myomectomy", "Osteoarthritis", "Osteoporosis",
             "Ovarian cancer", "Ovarian cysts", "Ovarian fibroma", "Overactive bladder", "Pancreatic cystic neoplasm", "Pap test", "Pelvic inflammatory disease (PID)",
             "Pelvic organ prolapse", "Pelvic pain", "Pelvic ultrasound", "Perimenopause", "Placenta accreta", "Placenta previa", "Polycystic ovary syndrome (PCOS)",
             "Postpartum depression", "Preconception planning", "Pregnancy", "Pregnancy complications", "Pregnancy, first trimester", "Pregnancy, multiples",
             "Pregnancy, second trimester", "Pregnancy, third trimester", "Premenstrual dysphoric disorder (PMDD)", "Premenstrual syndrome (PMS)", "Prenatal care", "Prenatal genetic screening", 
             "Preterm birth", "Primary care for women", "Recurrent pregnancy loss", "Rheumatoid arthritis", "Robotic hysterectomy", "Robotic myomectomy", "Sexually transmitted diseases (STDs)", 
             "Sleep disorders", "Small vessel disease", "Sports injuries", "STIs", "Stress urinary incontinence", "Stroke", "Thyroid disease", "Transgender care", "Tubal ligation",
             "Turner syndrome", "Type I autoimmune hepatitis", "Ultrasound", "Urinary incontinence", "Urinary tract infections (UTIs)", "Uterine and vaginal prolapse", "Uterine fibroids",
             "Vaginal birth after cesarean section", "Vaginal cancer", "Vaginitis", "Vulvar cancer", "Vulvar dysplasia", "Vulvitis", "Yeast infection"
]

Entrez.email = "sebrodnick@willamette.edu"  # this identifies who is pulling and lessen 429 errors for pulling

keywords_joined22 = " OR ".join(f'"{word}"[Title/abstract]' for word in keywords_df2)

start_date22 = datetime(1984, 1, 1)
end_date22 = datetime(2000, 12, 31)

# List to hold dfs for each day
dfs_list22 = []
# Loop through each day
current_date22 = start_date22

while current_date22 <= end_date22:
    last_day22 = calendar.monthrange(current_date22.year, current_date22.month)[1]
    start_str22 = current_date22.strftime("%Y/%m/01")
    end_str22 = current_date22.strftime(f"%Y/%m/{last_day22}")
    month_query22 = f'("{start_str22}"[Date - Create] : "{end_str22}"[Date - Create]) AND ({keywords_joined22})'

    try:
        df22 = get_pubmed_papers(query = month_query22, max_results = 9998)
        if df22 is not None and not df22.empty:
            dfs_list22.append(df22)
            print(f"Papers found for this year {start_str22} - {end_str22}: {len(df22)}")

        else:
            print(f"No papers found for this month {start_str22} - {end_str22}")
    except Exception as e:
        print(f"Error getting papers for {start_str22} - {end_str22}: {e}")

    time.sleep(3) # This will pause (3 seconds) between each loop, this will also help with 429 errors 
    
    if current_date22.month == 12:
        current_date22 = datetime(current_date22.year + 1, 1, 1)
    else:
        current_date22 = datetime(current_date22.year, current_date22.month + 1, 1)

if dfs_list22:
    final_df22 = pd.concat(dfs_list22, ignore_index=True)
    print(f"Total number of papers: {len(final_df22)}")

else:
    print("No data collected")
    final_df22 = pd.DataFrame()

Papers found for this year 1984/01/01 - 1984/01/31: 2489
Papers found for this year 1984/02/01 - 1984/02/29: 580
Papers found for this year 1984/03/01 - 1984/03/31: 760
Papers found for this year 1984/04/01 - 1984/04/30: 655
Papers found for this year 1984/05/01 - 1984/05/31: 664
Papers found for this year 1984/06/01 - 1984/06/30: 701
Papers found for this year 1984/07/01 - 1984/07/31: 716
Papers found for this year 1984/08/01 - 1984/08/31: 562
Papers found for this year 1984/09/01 - 1984/09/30: 750
Papers found for this year 1984/10/01 - 1984/10/31: 674
Papers found for this year 1984/11/01 - 1984/11/30: 665
Papers found for this year 1984/12/01 - 1984/12/31: 635
Papers found for this year 1985/01/01 - 1985/01/31: 2618
Papers found for this year 1985/02/01 - 1985/02/28: 633
Papers found for this year 1985/03/01 - 1985/03/31: 767
Papers found for this year 1985/04/01 - 1985/04/30: 625
Papers found for this year 1985/05/01 - 1985/05/31: 665
Papers found for this year 1985/06/01 - 1985/0

In [158]:
print(final_df22) 

                                                    title  \
0       [Extracranial-intracranial arterial anastomosi...   
1       Spatio-temporal processing in multiple sclerosis.   
2       Normotensive and spontaneously-hypertensive ra...   
3       [The change of fertility and factors impacting...   
4       [Limitations of the indicator-total fertility-...   
...                                                   ...   
270512  Avoiding multiple pregnancies in ART: multiple...   
270513  Risk-benefit decision making for treatment of ...   
270514  Does selenium reduce the risk of threatened pr...   
270515  Exposure to mercury in pregnant women from Alt...   
270516                    Management of male infertility.   

                                                 abstract  \
0                                                     NaN   
1       The processing of spatial and temporal detail ...   
2       The effect of arginine-vasopressin (AVP) on po...   
3       In 1981 a 3% ra

### Third time block: 2001 - 20206
This code started with this but errored. The script was changed to save each pull everytime into a csv, because the previous script was crashing. There are multiple different csvs:
- 2001 - 2006
- 2006 - 2009
- 2009 - 2024
- 2025 - 2026

In [15]:
# FOR FULL LIST OF KEYWORDS
# PER MONTH
###### 2001 - 2026
# Changed this script to make my computer not crash again

keywords_full = ["Abdominal cerclage", "Abnormal pap smears", "Abnormal uterine bleeding",  "Alzheimer's disease", "Amenorrhea",
            "Amniocentesis", "Arthritis", "Asthma", "Bacterial vaginosis", "Bleeding disorders", "Breast cancer", "Cancers",
            "Celiac disease", "Cervical cancer", "Cervical insufficiency", "Cervicitis", "Cesarean section (C-section)",
            "Chorionic villus sampling", "Colposcopy", "Congenital abnormalities", "Continence", "Contraception (birth control)",
            "Cystoscopy", "Depression and anxiety", "Dilation and curettage (D and C)", "Dysmenorrhea",  "Endometrial ablation",  "Endometrial biopsy",
            "Endometrial cancer", "Endometriosis", "Epilepsy", "Family planning", "Female genital cutting", "Fertility preservation",
            "Fibroma", "Gallstone disease",  "Genetic conditions", "Gentle cesarean section", "Gestational diabetes", "Gestational trophoblastic disease",
            "Graves' disease", "Gynecologic cancer", "Gynecologic specialty care", "Gynecologic well-woman care", "Gynecological surgery",  "Heart disease",
             "Heavy menstrual cycles", "High-risk pregnancy","HIV", "HPV", "In-vitro fertilization", "Infertility", "Integrative medicine", "Interstitial cystitis",
             "Irritable bowel syndrome (IBS)", "Laparoscopic hysterectomy", "Laparoscopy", "Loop electrosurgical excision procedure (LEEP)", "Lupus",
             "Menopause", "Menorrhagia", "Menstrual conditions", "Miscarriage", "Morning sickness", "Multiple sclerosis", "Myomectomy", "Osteoarthritis", "Osteoporosis",
             "Ovarian cancer", "Ovarian cysts", "Ovarian fibroma", "Overactive bladder", "Pancreatic cystic neoplasm", "Pap test", "Pelvic inflammatory disease (PID)",
             "Pelvic organ prolapse", "Pelvic pain", "Pelvic ultrasound", "Perimenopause", "Placenta accreta", "Placenta previa", "Polycystic ovary syndrome (PCOS)",
             "Postpartum depression", "Preconception planning", "Pregnancy", "Pregnancy complications", "Pregnancy, first trimester", "Pregnancy, multiples",
             "Pregnancy, second trimester", "Pregnancy, third trimester", "Premenstrual dysphoric disorder (PMDD)", "Premenstrual syndrome (PMS)", "Prenatal care", "Prenatal genetic screening", "Preterm birth",
             "Primary care for women", "Recurrent pregnancy loss", "Rheumatoid arthritis", "Robotic hysterectomy", "Robotic myomectomy", "Sexually transmitted diseases (STDs)", 
             "Sleep disorders", "Small vessel disease", "Sports injuries", "STIs", "Stress urinary incontinence", "Stroke", "Thyroid disease", "Transgender care", "Tubal ligation",
             "Turner syndrome", "Type I autoimmune hepatitis", "Ultrasound", "Urinary incontinence", "Urinary tract infections (UTIs)", "Uterine and vaginal prolapse", "Uterine fibroids",
             "Vaginal birth after cesarean section", "Vaginal cancer", "Vaginitis", "Vulvar cancer", "Vulvar dysplasia", "Vulvitis", "Yeast infection"
]

Entrez.email = "sebrodnick@willamette.edu"  

output_data = "raw_pubmed_2025_2026.csv"

# Before changing this it was 2001. So the data for 2001- 2026 is already in the raw_pubmed_data.csv
start_date11 = datetime(2025, 1, 1)
end_date11 = datetime(2026, 6, 27)


#dfs_list11 = []
current_date11 = start_date11

while current_date11 <= end_date11:
    last_day11 = calendar.monthrange(current_date11.year, current_date11.month)[1]
    start_str11 = current_date11.strftime("%Y/%m/01")
    end_str11 = current_date11.strftime(f"%Y/%m/{last_day11}")

    for keyword in keywords_full:
        month_query11 = f'("{start_str11}"[Date - Create] : "{end_str11}"[Date - Create]) AND ({keyword})'
        
        try:
            df11 = get_pubmed_papers(query = month_query11, max_results = 9998)
            
            if df11 is not None and not df11.empty:
                file_name = os.path.isfile(output_data)
                df11.to_csv(output_data, mode='a', header=not file_name, index=False)
                print(f"Saved papers for '{keyword}' for this year {start_str11} - {end_str11}: {len(df11)}")
                del df11  # Free up memory after saving to CSV
                
            else:
                print(f"No papers found for '{keyword}' this month {start_str11} - {end_str11}")
                
        except Exception as e:
            print(f"Error getting papers for '{keyword}': {start_str11} - {end_str11}: {e}")
            
        time.sleep(1) 
        
    if current_date11.month == 12:
        current_date11 = datetime(current_date11.year + 1, 1, 1)
        
    else:
        current_date11 = datetime(current_date11.year, current_date11.month + 1, 1)


Saved papers for 'Abdominal cerclage' for this year 2025/01/01 - 2025/01/31: 2
Saved papers for 'Abnormal pap smears' for this year 2025/01/01 - 2025/01/31: 6
Saved papers for 'Abnormal uterine bleeding' for this year 2025/01/01 - 2025/01/31: 34
Saved papers for 'Alzheimer's disease' for this year 2025/01/01 - 2025/01/31: 1615
Saved papers for 'Amenorrhea' for this year 2025/01/01 - 2025/01/31: 32
Saved papers for 'Amniocentesis' for this year 2025/01/01 - 2025/01/31: 23
INFO:pymed_paperscraper.api:ChunkedEncodingError: Response ended prematurely.	Now at 1 attempts for ['39768847', '39768826', '39768820', '39768619', '39768600', '39768575', '39768543', '39768539', '39768516', '39768486', '39768461', '39768307', '39768214', '39768138', '39767853', '39767648', '39767603', '39767447', '39767202', '39767180', '39766906', '39766566', '39766360', '39766343', '39766234', '39766233', '39766196', '39765965', '39765938', '39765601', '39765309', '39765268', '39765266', '39765191', '39765179', '39

In [ ]:
# This saved the data from the script above, so it got 2001 - 2026, well to october of 2026

#if 'dfs_list11' in locals() and dfs_list11:
    #df_saved_2 = pd.concat(dfs_list11, ignore_index = True)

In [2]:
# This is the final dataframe from the 5 total blocks
#pubmed_data = pd.concat([final_df1, final_df2, final_df, final_df22, df], ignore_index = True)

# Data Combination/Cleaning/Send to Warehouse

In [ ]:
# concat all dataframes into one final dataframe
# raw_pubmed
# raw_pubmed_2006_2009
# raw_pubmed_2009_2024
# raw_pubmed_2025_2026

In [7]:
df1 = pd.read_csv("/Users/shantibrodnick/Downloads/510Capstone/GenderDisparityinResearch/data/raw/raw_pubmed.csv")

In [8]:
df2 = pd.read_csv("/Users/shantibrodnick/Downloads/510Capstone/GenderDisparityinResearch/notebooks/Article_Data_APIs/raw_pubmed_2006_2009.csv")

In [9]:
df4 = pd.read_csv("/Users/shantibrodnick/Downloads/510Capstone/GenderDisparityinResearch/notebooks/Article_Data_APIs/raw_pubmed_2025_2026.csv")

In [ ]:
#################################################################################################################################################

In [ ]:
#df3 = pd.read_csv("/Users/shantibrodnick/Downloads/510Capstone/GenderDisparityinResearch/notebooks/Article_Data_APIs/raw_pubmed_2009_2024.csv")

# This dataframe is 18.89 GB. Here is is being scanned and compressed into a parquet file. This parquet file is only 3.75 GB.
pl.scan_csv("/Users/shantibrodnick/Downloads/510Capstone/GenderDisparityinResearch/notebooks/Article_Data_APIs/raw_pubmed_2009_2024.csv").sink_parquet("compressed_pubmed_2009_2024.parquet")

In [5]:
# Read in or scan parquet file
scanned_df = pl.scan_parquet("compressed_pubmed_2009_2024.parquet")

In [ ]:
# Check the amount of rows
scanned_df.select(pl.len()).collect().item()
# 10,881,939

In [13]:
# Drop duplicates
cleaned_df = scanned_df.unique(subset=["doi", "title"], keep="first")

In [14]:
# Check amount of rows after dropping duplicates
cleaned_df.select(pl.len()).collect().item()
# 6,711,691

6711691

In [14]:
# Parquet to csv
cleaned_df.sink_csv('cleaned_raw_pubmed_2009_2024.csv')

In [ ]:
#############################################################################################################################################################

In [5]:
df3 = pd.read_csv("/Users/shantibrodnick/Downloads/510Capstone/GenderDisparityinResearch/notebooks/Article_Data_APIs/cleaned_raw_pubmed_2009_2024.csv")

In [6]:
df3.info()
# 6,711,691

<class 'pandas.DataFrame'>
RangeIndex: 6711691 entries, 0 to 6711690
Data columns (total 6 columns):
 #   Column    Dtype
---  ------    -----
 0   title     str  
 1   abstract  str  
 2   journal   str  
 3   date      str  
 4   authors   str  
 5   doi       str  
dtypes: str(6)
memory usage: 10.8 GB


In [10]:
# Combine all the dfs 
raw_pubmed_data = pd.concat([df1, df2, df3, df4], ignore_index=True)

In [11]:
# View row count
raw_pubmed_data.info()
# 11,796,644

<class 'pandas.DataFrame'>
RangeIndex: 11796644 entries, 0 to 11796643
Data columns (total 6 columns):
 #   Column    Dtype
---  ------    -----
 0   title     str  
 1   abstract  str  
 2   journal   str  
 3   date      str  
 4   authors   str  
 5   doi       str  
dtypes: str(6)
memory usage: 18.2 GB


In [12]:
# Convert to datetime
raw_pubmed_data['date'] =  pd.to_datetime(raw_pubmed_data['date'], errors='coerce')

In [13]:
# Check dates
raw_pubmed_data['date'].min()

Timestamp('1781-06-01 00:00:00')

In [14]:
raw_pubmed_data['date'].max()

Timestamp('2026-07-01 00:00:00')

In [15]:
raw_pubmed_data['date'] = raw_pubmed_data['date'].dt.strftime('%Y-%m-%d')

In [ ]:
# Drop duplicates
raw_pubmed_data.drop_duplicates(subset = ['doi', 'title'], keep = 'first')
# THIS IS NOT WORKING BECAUSE IT IS SO BIG

: 

# Use parquet files to drop duplicates because the pubmed data is so large

In [ ]:
# raw pubmed data to parquet
#raw_pubmed_data.to_parquet("compressed_raw_pubmed_data.parquet")

In [10]:
final_df_maybe = pl.scan_parquet("/Users/shantibrodnick/Downloads/510Capstone/GenderDisparityinResearch/notebooks/Article_Data_APIs/Data Files/compressed_raw_pubmed_data.parquet")


In [11]:
final_df_maybe.select(pl.len()).collect().item()
# 11,796,644

11796644

In [12]:
# Yay it worked, now I can remove duplicates
cleaned_final_df = final_df_maybe.unique(subset=["doi", "title"], keep="first")

In [13]:
cleaned_final_df.select(pl.len()).collect().item()
# 10,206,788

10206788

In [17]:
cleaned_final_df.head(5).collect()

title,abstract,journal,date,authors,doi
str,str,str,str,str,str
"""Endoprosthetic treatment of a …","""To describe the successful ste…","""Journal of endovascular therap…","""2003-10-10""","""['Joren R GCallaert', 'IngeFou…","""10.1177/152660280301000424"""
"""Autodisplay of an endo-1,4-β-x…","""The goal of this work was the …","""Enzyme and microbial technolog…","""2021-07-28""","""['Victor EBalderas Hernández',…","""10.1016/j.enzmictec.2021.10983…"
"""Adjuvant radiotherapy in patie…","""Our study was to determine whe…","""Journal of cancer research and…","""2022-10-30""","""['HuanChen', 'MinQu', 'Haoqing…","""10.1007/s00432-022-04409-z"""
"""Cutaneous mucormycosis in a pa…","""We describe a patient with inv…","""The Journal of dermatology""","""2005-05-03""","""['HideakiMiyamoto', 'HiroyukiH…","""10.1111/j.1346-8138.2005.tb007…"
"""Study protocol: randomized con…","""Studies suggest that individua…","""BMC psychiatry""","""2024-03-27""","""['ElisabethJakob', 'JulianeMei…","""10.1186/s12888-024-05697-0"""


In [7]:
cleaned_final_df.sink_csv('cleaned_raw_pubmed_data.csv')

In [ ]:
# Read in csv
df = pd.read_csv("/Users/shantibrodnick/Downloads/510Capstone/GenderDisparityinResearch/notebooks/Article_Data_APIs/Data Files/cleaned_raw_pubmed_data.csv")

# Transfer final raw pubmed data to our warehouse

In [ ]:
##################################################################################################################################################
# RAW DATA TO DB
##################################################################################################################################################

In [9]:
# Connect to the database
conn = psycopg2.connect(
    dbname=os.getenv("DBNAME"),
    user=os.getenv("DBUSER"),
    password=os.getenv("DBPASSWORD"),
    port=os.getenv("DBPORT"),
    host=os.getenv("DBHOST")
)
conn_string=os.getenv("CONNSTRING")

# Create engine
engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DBUSER')}:{os.getenv('DBPASSWORD')}"
    f"@{os.getenv('DBHOST')}:{os.getenv('DBPORT')}/{os.getenv('DBNAME')}"
)

In [ ]:
#pubmed_data.to_sql("pubmed_articles", engine, if_exists="append", index=False, method="multi", chunksize=100)
df.to_sql("final_raw_pubmed_data", engine, if_exists="append", index=False, chunksize=100)

AttributeError: 'LazyFrame' object has no attribute 'to_sql'